# MuJoCo World Visualization

This notebook builds a MuJoCo-safe PR2 apartment-style world and visualizes it with `MujocoSim`. It does not execute PyCRAM actions; it only checks that the semantic digital twin world can be converted and simulated in MuJoCo.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

for rel_path in (
    "llmr_updated_arch/src",
    "pycram/src",
    "semantic_digital_twin/src",
    "physics_simulators/src",
    "uniworld/src",
):
    src_path = repo_root / rel_path
    if src_path.exists() and str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))

print("Repo root:", repo_root)

## Build A MuJoCo-Safe World

In [ ]:
from pycram.datastructures.dataclasses import Context
from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Milk, Cereal
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix
from semantic_digital_twin.world_description.connections import FixedConnection
from semantic_digital_twin.world_description.world_entity import Body

pycram_resources = repo_root / "pycram" / "resources"

PR2_URDF = "package://iai_pr2_description/robots/pr2_with_ft2_cableguide.xacro"
APARTMENT_URDF = str(pycram_resources / "worlds" / "apartment.urdf")
MILK_STL = str(pycram_resources / "objects" / "milk.stl")
CEREAL_STL = str(pycram_resources / "objects" / "breakfast_cereal.stl")


def build_mujoco_world():
    pr2_world = URDFParser.from_file(PR2_URDF).parse()
    PR2.from_world(pr2_world)

    apartment_world = URDFParser.from_file(APARTMENT_URDF).parse()
    pr2_world.merge_world(
        apartment_world,
        FixedConnection(
            parent=pr2_world.root,
            child=apartment_world.root,
        ),
    )

    apartment_root = pr2_world.get_body_by_name("apartment_root")

    milk_world = STLParser(MILK_STL).parse()
    pr2_world.merge_world(
        milk_world,
        FixedConnection(
            parent=apartment_root,
            child=milk_world.root,
            parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                2.37,
                2.0,
                1.05,
                reference_frame=apartment_root,
            ),
        ),
    )

    cereal_world = STLParser(CEREAL_STL).parse()
    pr2_world.merge_world(
        cereal_world,
        FixedConnection(
            parent=apartment_root,
            child=cereal_world.root,
            parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                2.37,
                1.8,
                1.05,
                reference_frame=apartment_root,
            ),
        ),
    )

    with pr2_world.modify_world():
        pr2_world.add_semantic_annotation(
            Milk(root=pr2_world.get_body_by_name("milk.stl"), _world=pr2_world)
        )
        pr2_world.add_semantic_annotation(
            Cereal(
                root=pr2_world.get_body_by_name("breakfast_cereal.stl"),
                _world=pr2_world,
            )
        )

    robot_view = pr2_world.get_semantic_annotations_by_type(PR2)[0]
    context = Context(pr2_world, robot_view)
    context.evaluate_conditions = False
    return pr2_world, robot_view, context


world, robot_view, context = build_mujoco_world()
symbol_type = Body

print("World loaded:", type(world).__name__)
print("Robot view  :", type(robot_view).__name__)
print("World root  :", world.root)

## Install Notebook-Local Mesh Conversion Patch

In [ ]:
from semantic_digital_twin.adapters import multi_sim


def safe_mujoco_mesh_post_convert(self, entity, shape_props, **kwargs):
    shape_props.update(
        multi_sim.MujocoGeomConverter._post_convert(
            self,
            entity,
            shape_props,
            **kwargs,
        )
    )
    shape_props.update({"mesh": entity})

    visual = getattr(entity.mesh, "visual", None)
    material = getattr(visual, "material", None) if visual is not None else None
    image = getattr(material, "image", None) if material is not None else None
    material_name = getattr(material, "name", None) if material is not None else None
    filename = getattr(image, "filename", None) if image is not None else None

    if (
        isinstance(visual, multi_sim.TextureVisuals)
        and isinstance(material_name, str)
        and isinstance(filename, str)
        and filename
    ):
        shape_props["texture_file_path"] = filename

    return shape_props


multi_sim.MujocoMeshConverter._post_convert = safe_mujoco_mesh_post_convert
MujocoSim = multi_sim.MujocoSim

print("Installed notebook-local MuJoCo mesh patch")

## Start MuJoCo

In [ ]:
import os

from physics_simulators.base_simulator import SimulatorConstraints

headless = os.environ.get("CI", "false").lower() == "true"

multi_sim = MujocoSim(
    world=world,
    headless=headless,
    step_size=0.001,
)

constraints = SimulatorConstraints(max_number_of_steps=20000)
multi_sim.start_simulation(constraints=constraints)

print("MuJoCo simulation started")
print("Headless:", headless)
print("Scene file:", multi_sim.default_file_path)

## Monitor Until Completion

In [ ]:
import time

time_start = time.time()

while multi_sim.is_running():
    time.sleep(0.1)
    if multi_sim.simulator.current_number_of_steps % 1000 == 0:
        print(
            "Current number of steps:",
            multi_sim.simulator.current_number_of_steps,
        )

print(f"Time elapsed: {time.time() - time_start:.2f}s")

## Stop MuJoCo

In [ ]:
if multi_sim.is_running():
    multi_sim.stop_simulation()
    print("MuJoCo simulation stopped")
else:
    print("Simulation already finished")